[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dongzoolee/hidden-bites/blob/main/notebooks/google-maps-review-word-frequency.ipynb)

# Google Maps 리뷰 자주 등장하는 단어 분석

위에서 크롤링한 `datasets/google-maps-reviews-2026-05-16/`의 장소별 JSON 중 하나를 raw GitHub URL로 불러와 리뷰 본문에서 자주 등장하는 단어를 확인합니다.

아래 URL 셀에서는 하나의 파일만 활성화되어 있고, 나머지 장소 파일은 주석 처리되어 있습니다. 다른 장소를 보고 싶으면 활성 줄을 바꾸면 됩니다.


## 0. 실행 환경 준비

한국어 단어를 조금 더 안정적으로 분리하기 위해 `kiwipiepy`를 사용합니다. 설치가 실패하거나 로컬에 없는 경우에도 정규식 기반 fallback으로 실행됩니다.


In [ ]:
%pip install -q kiwipiepy pandas matplotlib koreanize-matplotlib


## 1. 데이터 URL 선택

기본값은 1위 장소 파일입니다. 한 번에 여러 URL을 읽지 않도록 `DATASET_URL`은 하나만 켜두었습니다.


In [ ]:
# 1위 - 무탄 코엑스점 MUTAN COEX Store(coex mall food restaurants)ㅣ美食ㅣコエックスモール レストラン
DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/001-%E1%84%86%E1%85%AE%E1%84%90%E1%85%A1%E1%86%AB-%E1%84%8F%E1%85%A9%E1%84%8B%E1%85%A6%E1%86%A8%E1%84%89%E1%85%B3%E1%84%8C%E1%85%A5%E1%86%B7-mutan-coex-store-coex-mall-food-restaurants-%E1%85%B5%E7%BE%8E%E9%A3%9F%E1%85%B5%E3%82%B3%E3%82%A8%E3%83%83%E3%82%AF%E3%82%B9%E3%83%A2%E3%83%BC%E3%83%AB-%E3%83%AC%E3%82%B9%E3%83%88%E3%83%A9%E3%83%B3.json"

# 2위 - 핏제리아오 대학로본점
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/002-%E1%84%91%E1%85%B5%E1%86%BA%E1%84%8C%E1%85%A6%E1%84%85%E1%85%B5%E1%84%8B%E1%85%A1%E1%84%8B%E1%85%A9-%E1%84%83%E1%85%A2%E1%84%92%E1%85%A1%E1%86%A8%E1%84%85%E1%85%A9%E1%84%87%E1%85%A9%E1%86%AB%E1%84%8C%E1%85%A5%E1%86%B7.json"

# 3위 - 오다리집 간장게장
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/003-%E1%84%8B%E1%85%A9%E1%84%83%E1%85%A1%E1%84%85%E1%85%B5%E1%84%8C%E1%85%B5%E1%86%B8-%E1%84%80%E1%85%A1%E1%86%AB%E1%84%8C%E1%85%A1%E1%86%BC%E1%84%80%E1%85%A6%E1%84%8C%E1%85%A1%E1%86%BC.json"

# 4위 - 홍대 맛집 깃뜰 | Hongdae Korean BBQ git teul | サムギョプサル | 美食 | 烤肉
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/004-%E1%84%92%E1%85%A9%E1%86%BC%E1%84%83%E1%85%A2-%E1%84%86%E1%85%A1%E1%86%BA%E1%84%8C%E1%85%B5%E1%86%B8-%E1%84%80%E1%85%B5%E1%86%BA%E1%84%84%E1%85%B3%E1%86%AF-hongdae-korean-bbq-git-teul-%E3%82%B5%E3%83%A0%E3%82%AD-%E3%83%A7%E3%83%95-%E3%82%B5%E3%83%AB-%E7%BE%8E%E9%A3%9F-%E7%83%A4%E8%82%89.json"

# 5위 - 강남 돼지상회 무한리필 홍대점 | Hongdae Korean bbq restaurant All you can eat | サムギョプサル | 焼肉 | 烤肉
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/005-%E1%84%80%E1%85%A1%E1%86%BC%E1%84%82%E1%85%A1%E1%86%B7-%E1%84%83%E1%85%AB%E1%84%8C%E1%85%B5%E1%84%89%E1%85%A1%E1%86%BC%E1%84%92%E1%85%AC-%E1%84%86%E1%85%AE%E1%84%92%E1%85%A1%E1%86%AB%E1%84%85%E1%85%B5%E1%84%91%E1%85%B5%E1%86%AF-%E1%84%92%E1%85%A9%E1%86%BC%E1%84%83%E1%85%A2%E1%84%8C%E1%85%A5%E1%86%B7-hongdae-korean-bbq-restaurant-all-you-can-e.json"

# 6위 - Mongvely Myeongdong Korean BBQ Beef All You Can Eat | 焼き肉 焼肉 | 烤肉 l 무한리필 몽블리 명동점
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/006-mongvely-myeongdong-korean-bbq-beef-all-you-can-eat-%E7%84%BC%E3%81%8D%E8%82%89-%E7%84%BC%E8%82%89-%E7%83%A4%E8%82%89-l-%E1%84%86%E1%85%AE%E1%84%92%E1%85%A1%E1%86%AB%E1%84%85%E1%85%B5%E1%84%91%E1%85%B5%E1%86%AF-%E1%84%86%E1%85%A9%E1%86%BC%E1%84%87%E1%85%B3.json"

# 7위 - 왕비집 명동본점
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/007-%E1%84%8B%E1%85%AA%E1%86%BC%E1%84%87%E1%85%B5%E1%84%8C%E1%85%B5%E1%86%B8-%E1%84%86%E1%85%A7%E1%86%BC%E1%84%83%E1%85%A9%E1%86%BC%E1%84%87%E1%85%A9%E1%86%AB%E1%84%8C%E1%85%A5%E1%86%B7.json"

# 8위 - 무교동북어국집
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/008-%E1%84%86%E1%85%AE%E1%84%80%E1%85%AD%E1%84%83%E1%85%A9%E1%86%BC%E1%84%87%E1%85%AE%E1%86%A8%E1%84%8B%E1%85%A5%E1%84%80%E1%85%AE%E1%86%A8%E1%84%8C%E1%85%B5%E1%86%B8.json"

# 9위 - 미도갈비
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/009-%E1%84%86%E1%85%B5%E1%84%83%E1%85%A9%E1%84%80%E1%85%A1%E1%86%AF%E1%84%87%E1%85%B5.json"

# 10위 - 육지 홍대
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/010-%E1%84%8B%E1%85%B2%E1%86%A8%E1%84%8C%E1%85%B5-%E1%84%92%E1%85%A9%E1%86%BC%E1%84%83%E1%85%A2.json"

# 11위 - 보광정 이태원점
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/011-%E1%84%87%E1%85%A9%E1%84%80%E1%85%AA%E1%86%BC%E1%84%8C%E1%85%A5%E1%86%BC-%E1%84%8B%E1%85%B5%E1%84%90%E1%85%A2%E1%84%8B%E1%85%AF%E1%86%AB%E1%84%8C%E1%85%A5%E1%86%B7.json"

# 12위 - 육몽 홍대본점
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/012-%E1%84%8B%E1%85%B2%E1%86%A8%E1%84%86%E1%85%A9%E1%86%BC-%E1%84%92%E1%85%A9%E1%86%BC%E1%84%83%E1%85%A2%E1%84%87%E1%85%A9%E1%86%AB%E1%84%8C%E1%85%A5%E1%86%B7.json"

# 13위 - 이태리국시 성수(ITALYGUKSI SEONGSU)
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/013-%E1%84%8B%E1%85%B5%E1%84%90%E1%85%A2%E1%84%85%E1%85%B5%E1%84%80%E1%85%AE%E1%86%A8%E1%84%89%E1%85%B5-%E1%84%89%E1%85%A5%E1%86%BC%E1%84%89%E1%85%AE-italyguksi-seongsu.json"

# 14위 - 서울맛집 지강한식당 압구정본점 | restaurants | 韓国ナッコプセレストラン | 餐馆
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/014-%E1%84%89%E1%85%A5%E1%84%8B%E1%85%AE%E1%86%AF%E1%84%86%E1%85%A1%E1%86%BA%E1%84%8C%E1%85%B5%E1%86%B8-%E1%84%8C%E1%85%B5%E1%84%80%E1%85%A1%E1%86%BC%E1%84%92%E1%85%A1%E1%86%AB%E1%84%89%E1%85%B5%E1%86%A8%E1%84%83%E1%85%A1%E1%86%BC-%E1%84%8B%E1%85%A1%E1%86%B8%E1%84%80%E1%85%AE%E1%84%8C%E1%85%A5%E1%86%BC%E1%84%87%E1%85%A9%E1%86%AB%E1%84%8C%E1%85%A5%E1%86%B7-restaurants-%E9%9F%93%E5%9B%BD%E3%83%8A%E3%83%83%E3%82%B3%E3%83%95-%E3%82%BB%E3%83%AC%E3%82%B9%E3%83%88%E3%83%A9%E3%83%B3-%E9%A4%90%E9%A6%86.json"

# 15위 - 쌤쌤쌤
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/015-%E1%84%8A%E1%85%A2%E1%86%B7%E1%84%8A%E1%85%A2%E1%86%B7%E1%84%8A%E1%85%A2%E1%86%B7.json"

# 16위 - 이국도산 EEGUK
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/016-%E1%84%8B%E1%85%B5%E1%84%80%E1%85%AE%E1%86%A8%E1%84%83%E1%85%A9%E1%84%89%E1%85%A1%E1%86%AB-eeguk.json"

# 17위 - 농민백암순대 본점
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/017-%E1%84%82%E1%85%A9%E1%86%BC%E1%84%86%E1%85%B5%E1%86%AB%E1%84%87%E1%85%A2%E1%86%A8%E1%84%8B%E1%85%A1%E1%86%B7%E1%84%89%E1%85%AE%E1%86%AB%E1%84%83%E1%85%A2-%E1%84%87%E1%85%A9%E1%86%AB%E1%84%8C%E1%85%A5%E1%86%B7.json"

# 18위 - 새마을식당 홍대서교점
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/018-%E1%84%89%E1%85%A2%E1%84%86%E1%85%A1%E1%84%8B%E1%85%B3%E1%86%AF%E1%84%89%E1%85%B5%E1%86%A8%E1%84%83%E1%85%A1%E1%86%BC-%E1%84%92%E1%85%A9%E1%86%BC%E1%84%83%E1%85%A2%E1%84%89%E1%85%A5%E1%84%80%E1%85%AD%E1%84%8C%E1%85%A5%E1%86%B7.json"

# 19위 - 명동역 닭갈비 맛집 | 장인닭갈비 명동점 | Myeongdong JangIn Dakgalbi Restaurants | 匠人铁板鸡 明洞店 | 職人タッカルビ 明洞
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/019-%E1%84%86%E1%85%A7%E1%86%BC%E1%84%83%E1%85%A9%E1%86%BC%E1%84%8B%E1%85%A7%E1%86%A8-%E1%84%83%E1%85%A1%E1%86%B0%E1%84%80%E1%85%A1%E1%86%AF%E1%84%87%E1%85%B5-%E1%84%86%E1%85%A1%E1%86%BA%E1%84%8C%E1%85%B5%E1%86%B8-%E1%84%8C%E1%85%A1%E1%86%BC%E1%84%8B%E1%85%B5%E1%86%AB%E1%84%83%E1%85%A1%E1%86%B0%E1%84%80%E1%85%A1%E1%86%AF%E1%84%87%E1%85%B5-%E1%84%86%E1%85%A7%E1%86%BC%E1%84%83%E1%85%A9%E1%86%BC%E1%84%8C%E1%85%A5%E1%86%B7-myeongdong-jangin-dakgalbi-re.json"

# 20위 - 성수다락 | 성수 레스토랑 | Seongsu Restaurants | 성수 맛집 | レストラン | 聖水洞美食
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/020-%E1%84%89%E1%85%A5%E1%86%BC%E1%84%89%E1%85%AE%E1%84%83%E1%85%A1%E1%84%85%E1%85%A1%E1%86%A8-%E1%84%89%E1%85%A5%E1%86%BC%E1%84%89%E1%85%AE-%E1%84%85%E1%85%A6%E1%84%89%E1%85%B3%E1%84%90%E1%85%A9%E1%84%85%E1%85%A1%E1%86%BC-seongsu-restaurants-%E1%84%89%E1%85%A5%E1%86%BC%E1%84%89%E1%85%AE-%E1%84%86%E1%85%A1%E1%86%BA%E1%84%8C%E1%85%B5%E1%86%B8-%E3%83%AC%E3%82%B9%E3%83%88%E3%83%A9%E3%83%B3-%E8%81%96%E6%B0%B4%E6%B4%9E%E7%BE%8E%E9%A3%9F.json"

# 21위 - 탐광
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/021-%E1%84%90%E1%85%A1%E1%86%B7%E1%84%80%E1%85%AA%E1%86%BC.json"

# 22위 - 왕비집 명동중앙점
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/022-%E1%84%8B%E1%85%AA%E1%86%BC%E1%84%87%E1%85%B5%E1%84%8C%E1%85%B5%E1%86%B8-%E1%84%86%E1%85%A7%E1%86%BC%E1%84%83%E1%85%A9%E1%86%BC%E1%84%8C%E1%85%AE%E1%86%BC%E1%84%8B%E1%85%A1%E1%86%BC%E1%84%8C%E1%85%A5%E1%86%B7.json"

# 23위 - 스케줄
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/023-%E1%84%89%E1%85%B3%E1%84%8F%E1%85%A6%E1%84%8C%E1%85%AE%E1%86%AF.json"

# 24위 - 한식왕비집 을지로점
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/024-%E1%84%92%E1%85%A1%E1%86%AB%E1%84%89%E1%85%B5%E1%86%A8%E1%84%8B%E1%85%AA%E1%86%BC%E1%84%87%E1%85%B5%E1%84%8C%E1%85%B5%E1%86%B8-%E1%84%8B%E1%85%B3%E1%86%AF%E1%84%8C%E1%85%B5%E1%84%85%E1%85%A9%E1%84%8C%E1%85%A5%E1%86%B7.json"

# 25위 - 솥내음 마곡 발산역점
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/025-%E1%84%89%E1%85%A9%E1%87%80%E1%84%82%E1%85%A2%E1%84%8B%E1%85%B3%E1%86%B7-%E1%84%86%E1%85%A1%E1%84%80%E1%85%A9%E1%86%A8-%E1%84%87%E1%85%A1%E1%86%AF%E1%84%89%E1%85%A1%E1%86%AB%E1%84%8B%E1%85%A7%E1%86%A8%E1%84%8C%E1%85%A5%E1%86%B7.json"

# 26위 - 별양집
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/026-%E1%84%87%E1%85%A7%E1%86%AF%E1%84%8B%E1%85%A3%E1%86%BC%E1%84%8C%E1%85%B5%E1%86%B8.json"

# 27위 - 월화고기 보라매점
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/027-%E1%84%8B%E1%85%AF%E1%86%AF%E1%84%92%E1%85%AA%E1%84%80%E1%85%A9%E1%84%80%E1%85%B5-%E1%84%87%E1%85%A9%E1%84%85%E1%85%A1%E1%84%86%E1%85%A2%E1%84%8C%E1%85%A5%E1%86%B7.json"

# 28위 - 마포곱창타운
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/028-%E1%84%86%E1%85%A1%E1%84%91%E1%85%A9%E1%84%80%E1%85%A9%E1%86%B8%E1%84%8E%E1%85%A1%E1%86%BC%E1%84%90%E1%85%A1%E1%84%8B%E1%85%AE%E1%86%AB.json"

# 29위 - 곱 마포직영점 공덕맛집ㅣmapo gopchang restaurantㅣ コプチャン | ホルモン | 孔德站 美食|대창|막창|회식|모임
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/029-%E1%84%80%E1%85%A9%E1%86%B8-%E1%84%86%E1%85%A1%E1%84%91%E1%85%A9%E1%84%8C%E1%85%B5%E1%86%A8%E1%84%8B%E1%85%A7%E1%86%BC%E1%84%8C%E1%85%A5%E1%86%B7-%E1%84%80%E1%85%A9%E1%86%BC%E1%84%83%E1%85%A5%E1%86%A8%E1%84%86%E1%85%A1%E1%86%BA%E1%84%8C%E1%85%B5%E1%86%B8%E1%85%B5mapo-gopchang-restaurant%E1%85%B5-%E3%82%B3%E3%83%95-%E3%83%81%E3%83%A3%E3%83%B3-%E3%83%9B%E3%83%AB%E3%83%A2%E3%83%B3-%E5%AD%94%E5%BE%B7%E7%AB%99-%E7%BE%8E%E9%A3%9F-%E1%84%83%E1%85%A2%E1%84%8E%E1%85%A1.json"

# 30위 - 신림춘천집 구로디지털직영점
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/030-%E1%84%89%E1%85%B5%E1%86%AB%E1%84%85%E1%85%B5%E1%86%B7%E1%84%8E%E1%85%AE%E1%86%AB%E1%84%8E%E1%85%A5%E1%86%AB%E1%84%8C%E1%85%B5%E1%86%B8-%E1%84%80%E1%85%AE%E1%84%85%E1%85%A9%E1%84%83%E1%85%B5%E1%84%8C%E1%85%B5%E1%84%90%E1%85%A5%E1%86%AF%E1%84%8C%E1%85%B5%E1%86%A8%E1%84%8B%E1%85%A7%E1%86%BC%E1%84%8C%E1%85%A5%E1%86%B7.json"

# 31위 - 오레노라멘 본점
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/031-%E1%84%8B%E1%85%A9%E1%84%85%E1%85%A6%E1%84%82%E1%85%A9%E1%84%85%E1%85%A1%E1%84%86%E1%85%A6%E1%86%AB-%E1%84%87%E1%85%A9%E1%86%AB%E1%84%8C%E1%85%A5%E1%86%B7.json"

# 32위 - 왕비집 종로점
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/032-%E1%84%8B%E1%85%AA%E1%86%BC%E1%84%87%E1%85%B5%E1%84%8C%E1%85%B5%E1%86%B8-%E1%84%8C%E1%85%A9%E1%86%BC%E1%84%85%E1%85%A9%E1%84%8C%E1%85%A5%E1%86%B7.json"

# 33위 - 돼지래스토랑 둘째
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/033-%E1%84%83%E1%85%AB%E1%84%8C%E1%85%B5%E1%84%85%E1%85%A2%E1%84%89%E1%85%B3%E1%84%90%E1%85%A9%E1%84%85%E1%85%A1%E1%86%BC-%E1%84%83%E1%85%AE%E1%86%AF%E1%84%8D%E1%85%A2.json"

# 34위 - 멘쇼쿠
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/034-%E1%84%86%E1%85%A6%E1%86%AB%E1%84%89%E1%85%AD%E1%84%8F%E1%85%AE.json"

# 35위 - 송파구맛집 지강한식당 잠실점 | restaurants | ナッコプセ | チキンカルビ レストラン | 鸡排
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/035-%E1%84%89%E1%85%A9%E1%86%BC%E1%84%91%E1%85%A1%E1%84%80%E1%85%AE%E1%84%86%E1%85%A1%E1%86%BA%E1%84%8C%E1%85%B5%E1%86%B8-%E1%84%8C%E1%85%B5%E1%84%80%E1%85%A1%E1%86%BC%E1%84%92%E1%85%A1%E1%86%AB%E1%84%89%E1%85%B5%E1%86%A8%E1%84%83%E1%85%A1%E1%86%BC-%E1%84%8C%E1%85%A1%E1%86%B7%E1%84%89%E1%85%B5%E1%86%AF%E1%84%8C%E1%85%A5%E1%86%B7-restaurants-%E3%83%8A%E3%83%83%E3%82%B3%E3%83%95-%E3%82%BB-%E3%83%81%E3%82%AD%E3%83%B3%E3%82%AB%E3%83%AB%E3%83%92-%E3%83%AC%E3%82%B9%E3%83%88%E3%83%A9%E3%83%B3-%E9%B8%A1%E6%8E%92.json"

# 36위 - 명동맛집 더식당 Myeongdong Korea bbq Restaurants THE SIC DDANG | Kfood | kbbq | 明洞 グルメ 韓国焼肉レストラン | 明洞必食
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/036-%E1%84%86%E1%85%A7%E1%86%BC%E1%84%83%E1%85%A9%E1%86%BC%E1%84%86%E1%85%A1%E1%86%BA%E1%84%8C%E1%85%B5%E1%86%B8-%E1%84%83%E1%85%A5%E1%84%89%E1%85%B5%E1%86%A8%E1%84%83%E1%85%A1%E1%86%BC-myeongdong-korea-bbq-restaurants-the-sic-ddang-kfood-kbbq-.json"

# 37위 - 혼고집 명동직영점
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/037-%E1%84%92%E1%85%A9%E1%86%AB%E1%84%80%E1%85%A9%E1%84%8C%E1%85%B5%E1%86%B8-%E1%84%86%E1%85%A7%E1%86%BC%E1%84%83%E1%85%A9%E1%86%BC%E1%84%8C%E1%85%B5%E1%86%A8%E1%84%8B%E1%85%A7%E1%86%BC%E1%84%8C%E1%85%A5%E1%86%B7.json"

# 38위 - 빠리가옥
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/038-%E1%84%88%E1%85%A1%E1%84%85%E1%85%B5%E1%84%80%E1%85%A1%E1%84%8B%E1%85%A9%E1%86%A8.json"

# 39위 - Myeongdong Korean BBQ 태초갈비 명동점 | Taecho Premium Beef 1++ | 明洞烤肉 | 明洞美食 烤肉 | ユッケビビンバ
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/039-myeongdong-korean-bbq-%E1%84%90%E1%85%A2%E1%84%8E%E1%85%A9%E1%84%80%E1%85%A1%E1%86%AF%E1%84%87%E1%85%B5-%E1%84%86%E1%85%A7%E1%86%BC%E1%84%83%E1%85%A9%E1%86%BC%E1%84%8C%E1%85%A5%E1%86%B7-taecho-premium-beef-1-%E6%98%8E%E6%B4%9E%E7%83%A4%E8%82%89-%E6%98%8E%E6%B4%9E%E7%BE%8E%E9%A3%9F-%E7%83%A4%E8%82%89-%E3%83%A6%E3%83%83%E3%82%B1.json"

# 40위 - 오시 망원본점
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/040-%E1%84%8B%E1%85%A9%E1%84%89%E1%85%B5-%E1%84%86%E1%85%A1%E1%86%BC%E1%84%8B%E1%85%AF%E1%86%AB%E1%84%87%E1%85%A9%E1%86%AB%E1%84%8C%E1%85%A5%E1%86%B7.json"

# 41위 - 하이웨이 서울 기사식당 영등포 타임스퀘어점 HIGHWAY SEOUL Restaurant タイムズスクエアのレストラン 海威餐厅 (永登浦店）
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/041-%E1%84%92%E1%85%A1%E1%84%8B%E1%85%B5%E1%84%8B%E1%85%B0%E1%84%8B%E1%85%B5-%E1%84%89%E1%85%A5%E1%84%8B%E1%85%AE%E1%86%AF-%E1%84%80%E1%85%B5%E1%84%89%E1%85%A1%E1%84%89%E1%85%B5%E1%86%A8%E1%84%83%E1%85%A1%E1%86%BC-%E1%84%8B%E1%85%A7%E1%86%BC%E1%84%83%E1%85%B3%E1%86%BC%E1%84%91%E1%85%A9-%E1%84%90%E1%85%A1%E1%84%8B%E1%85%B5%E1%86%B7%E1%84%89%E1%85%B3%E1%84%8F%E1%85%B0%E1%84%8B%E1%85%A5%E1%84%8C%E1%85%A5%E1%86%B7-highway-seoul-restaurant-%E3%82%BF%E3%82%A4%E3%83%A0%E3%82%B9-.json"

# 42위 - 농민백암순대
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/042-%E1%84%82%E1%85%A9%E1%86%BC%E1%84%86%E1%85%B5%E1%86%AB%E1%84%87%E1%85%A2%E1%86%A8%E1%84%8B%E1%85%A1%E1%86%B7%E1%84%89%E1%85%AE%E1%86%AB%E1%84%83%E1%85%A2.json"

# 43위 - 유즈라멘 본점
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/043-%E1%84%8B%E1%85%B2%E1%84%8C%E1%85%B3%E1%84%85%E1%85%A1%E1%84%86%E1%85%A6%E1%86%AB-%E1%84%87%E1%85%A9%E1%86%AB%E1%84%8C%E1%85%A5%E1%86%B7.json"

# 44위 - 칠프로칠백식당 신논현점
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/044-%E1%84%8E%E1%85%B5%E1%86%AF%E1%84%91%E1%85%B3%E1%84%85%E1%85%A9%E1%84%8E%E1%85%B5%E1%86%AF%E1%84%87%E1%85%A2%E1%86%A8%E1%84%89%E1%85%B5%E1%86%A8%E1%84%83%E1%85%A1%E1%86%BC-%E1%84%89%E1%85%B5%E1%86%AB%E1%84%82%E1%85%A9%E1%86%AB%E1%84%92%E1%85%A7%E1%86%AB%E1%84%8C%E1%85%A5%E1%86%B7.json"

# 45위 - 빤닭빤닭(bbandak)닭갈비 Dakgalbi 닭한마리 명동맛집 / 明洞 グルメ / 明洞 タッカンマリ/ best restaurant /good restaurant
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/045-%E1%84%88%E1%85%A1%E1%86%AB%E1%84%83%E1%85%A1%E1%86%B0%E1%84%88%E1%85%A1%E1%86%AB%E1%84%83%E1%85%A1%E1%86%B0-bbandak-%E1%84%83%E1%85%A1%E1%86%B0%E1%84%80%E1%85%A1%E1%86%AF%E1%84%87%E1%85%B5-dakgalbi-%E1%84%83%E1%85%A1%E1%86%B0%E1%84%92%E1%85%A1%E1%86%AB%E1%84%86%E1%85%A1%E1%84%85%E1%85%B5-%E1%84%86%E1%85%A7%E1%86%BC%E1%84%83%E1%85%A9%E1%86%BC%E1%84%86%E1%85%A1%E1%86%BA%E1%84%8C%E1%85%B5%E1%86%B8-%E6%98%8E%E6%B4%9E-%E3%82%AF-%E3%83%AB%E3%83%A1-%E6%98%8E%E6%B4%9E-%E3%82%BF%E3%83%83%E3%82%AB%E3%83%B3%E3%83%9E%E3%83%AA.json"

# 46위 - 아비꼬 타임스퀘어점
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/046-%E1%84%8B%E1%85%A1%E1%84%87%E1%85%B5%E1%84%81%E1%85%A9-%E1%84%90%E1%85%A1%E1%84%8B%E1%85%B5%E1%86%B7%E1%84%89%E1%85%B3%E1%84%8F%E1%85%B0%E1%84%8B%E1%85%A5%E1%84%8C%E1%85%A5%E1%86%B7.json"

# 47위 - 아베크 청담 Avecque Cheongdam Korean Italian K-Fusion
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/047-%E1%84%8B%E1%85%A1%E1%84%87%E1%85%A6%E1%84%8F%E1%85%B3-%E1%84%8E%E1%85%A5%E1%86%BC%E1%84%83%E1%85%A1%E1%86%B7-avecque-cheongdam-korean-italian-k-fusion.json"

# 48위 - 다몽집 | damongzip
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/048-%E1%84%83%E1%85%A1%E1%84%86%E1%85%A9%E1%86%BC%E1%84%8C%E1%85%B5%E1%86%B8-damongzip.json"

# 49위 - 뚝배기집
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/049-%E1%84%84%E1%85%AE%E1%86%A8%E1%84%87%E1%85%A2%E1%84%80%E1%85%B5%E1%84%8C%E1%85%B5%E1%86%B8.json"

# 50위 - 원조할아버지손두부
# DATASET_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/refs/heads/main/datasets/google-maps-reviews-2026-05-16/050-%E1%84%8B%E1%85%AF%E1%86%AB%E1%84%8C%E1%85%A9%E1%84%92%E1%85%A1%E1%86%AF%E1%84%8B%E1%85%A1%E1%84%87%E1%85%A5%E1%84%8C%E1%85%B5%E1%84%89%E1%85%A9%E1%86%AB%E1%84%83%E1%85%AE%E1%84%87%E1%85%AE.json"


## 2. JSON 로드와 리뷰 테이블 구성

각 JSON은 `metadata`와 `reviews`를 가지고 있습니다. 분석에는 리뷰 본문 `text`를 우선 사용하고, 본문이 비어 있으면 `raw_text`를 보조로 사용합니다.


In [ ]:
import json
import re
import unicodedata
from collections import Counter, defaultdict
from urllib.request import urlopen

import matplotlib.pyplot as plt
import pandas as pd

try:
    import koreanize_matplotlib
except Exception:
    pass
from IPython.display import display

try:
    from kiwipiepy import Kiwi
except Exception:
    Kiwi = None

with urlopen(DATASET_URL) as response:
    dataset = json.loads(response.read().decode("utf-8"))

metadata = dataset.get("metadata", {})
reviews = dataset.get("reviews", [])
rows = []
for index, review in enumerate(reviews, start=1):
    text = review.get("text") or review.get("raw_text") or ""
    text = unicodedata.normalize("NFC", str(text)).strip()
    if text:
        rows.append({
            "row": index,
            "place_rank": review.get("place_rank") or metadata.get("place_rank"),
            "place_name": review.get("place_name") or metadata.get("place", {}).get("displayName", {}).get("text"),
            "rating": review.get("rating"),
            "relative_time": review.get("relative_time"),
            "text": text,
        })

review_df = pd.DataFrame(rows)
summary = {
    "url": DATASET_URL,
    "place_rank": metadata.get("place_rank"),
    "place_name": metadata.get("place", {}).get("displayName", {}).get("text"),
    "status": metadata.get("status"),
    "target_reviews": metadata.get("target_reviews"),
    "loaded_reviews": len(reviews),
    "non_empty_text_reviews": len(review_df),
}
display(pd.DataFrame([summary]))
display(review_df.head(10))


## 3. 단어 추출

명사, 동사, 형용사, 어근, 외국어 토큰을 중심으로 뽑고, 너무 흔한 조사성 표현과 분석에 도움이 적은 단어는 제외합니다. `total_count`는 전체 등장 횟수, `review_count`는 해당 단어가 등장한 리뷰 수입니다.


In [ ]:
STOPWORDS = {
    "그리고", "그래서", "하지만", "근데", "정말", "진짜", "너무", "아주", "완전", "매우",
    "조금", "약간", "계속", "다시", "바로", "그냥", "여기", "저기", "이곳", "이번",
    "오늘", "어제", "내일", "우리", "제가", "저희", "제가", "것도", "있는", "없는",
    "해서", "하고", "하면", "보다", "같이", "같은", "때문", "정도", "리뷰", "방문",
    "place", "google", "maps", "review", "restaurant", "seoul",
}

kiwi = Kiwi() if Kiwi is not None else None
allowed_prefixes = ("NN", "VV", "VA", "XR", "SL")

def fallback_tokenize(text):
    candidates = re.findall(r"[가-힣A-Za-z]{2,}", text.lower())
    return [token for token in candidates if token not in STOPWORDS]

def tokenize(text):
    normalized = unicodedata.normalize("NFC", str(text))
    if kiwi is None:
        return fallback_tokenize(normalized)
    tokens = []
    for token in kiwi.tokenize(normalized):
        form = token.form.strip().lower()
        if len(form) < 2:
            continue
        if form in STOPWORDS:
            continue
        if token.tag.startswith(allowed_prefixes):
            tokens.append(form)
    return tokens

token_counter = Counter()
review_counter = Counter()
examples = defaultdict(list)

for row in review_df.to_dict("records"):
    tokens = tokenize(row["text"])
    token_counter.update(tokens)
    for token in set(tokens):
        review_counter[token] += 1
        if len(examples[token]) < 3:
            snippet = row["text"].replace("\n", " ")[:140]
            examples[token].append(snippet)

word_rows = [
    {
        "word": word,
        "total_count": count,
        "review_count": review_counter[word],
        "review_share": review_counter[word] / len(review_df) if len(review_df) else 0,
        "examples": " / ".join(examples[word]),
    }
    for word, count in token_counter.most_common()
]

word_df = pd.DataFrame(word_rows)
display(word_df.head(100))


## 4. 상위 단어 시각화

상위 단어를 전체 등장 횟수 기준으로 봅니다. 특정 단어가 한 리뷰 안에서 반복되는 효과를 줄이고 싶으면 `review_count` 기준으로 정렬하면 됩니다.


In [ ]:
plot_df = word_df.head(30).sort_values("total_count", ascending=True)
plt.rcParams["axes.unicode_minus"] = False
plt.figure(figsize=(10, 9))
plt.barh(plot_df["word"], plot_df["total_count"], color="#376f6b")
plt.xlabel("등장 횟수")
plt.title("자주 등장하는 단어 Top 30")
plt.tight_layout()
plt.show()


## 5. 자주 등장하는 2단어 조합

단어 하나만 보면 `맛있`, `친절`, `메뉴`처럼 넓은 표현이 많습니다. 연속된 두 단어 조합을 같이 보면 어떤 맥락에서 자주 쓰였는지 조금 더 잘 보입니다.


In [ ]:
bigram_counter = Counter()
bigram_review_counter = Counter()

for text in review_df["text"].tolist():
    tokens = tokenize(text)
    bigrams = [f"{left} {right}" for left, right in zip(tokens, tokens[1:])]
    bigram_counter.update(bigrams)
    bigram_review_counter.update(set(bigrams))

bigram_rows = [
    {
        "bigram": bigram,
        "total_count": count,
        "review_count": bigram_review_counter[bigram],
        "review_share": bigram_review_counter[bigram] / len(review_df) if len(review_df) else 0,
    }
    for bigram, count in bigram_counter.most_common(100)
]

bigram_df = pd.DataFrame(bigram_rows)
display(bigram_df)


## 6. 결과 저장용 CSV 만들기

Colab에서 실행했다면 아래 셀로 단어 빈도표와 2단어 조합 빈도표를 CSV로 저장할 수 있습니다.


In [ ]:
word_df.to_csv("google_maps_review_word_frequency.csv", index=False, encoding="utf-8-sig")
bigram_df.to_csv("google_maps_review_bigram_frequency.csv", index=False, encoding="utf-8-sig")
print("saved google_maps_review_word_frequency.csv")
print("saved google_maps_review_bigram_frequency.csv")
